# Debugging Truncated Filename Generation

This notebook investigates why regenerating a thumbnail results in a truncated filename like `0130.jpg` instead of the expected `0130_1800-晚報YT縮圖1-美伊開戰航母.jpg`.

## 1. Setup: Mock File Data and Variables
Define the expected and erroneous filenames for testing.

In [1]:
import os
import re

# The filename that resulted from the bug
incorrect_filename = "0130.jpg"

# The filename we expect
expected_filename = "0130_1800-晚報YT縮圖1-美伊開戰航母.jpg"

# The example file path structure
mock_file_path = r"c:\Users\cgadmin\晚報YT縮圖\1800-晚報YT縮圖1-美伊開戰航母.txt" # Hypothetical source file
mock_slug = "1800-晚報YT縮圖1-美伊開戰航母"
mock_date = "0130"

print(f"Goal: Generate '{expected_filename}' from '{mock_file_path}' or slug '{mock_slug}'")

Goal: Generate '0130_1800-晚報YT縮圖1-美伊開戰航母.jpg' from 'c:\Users\cgadmin\晚報YT縮圖\1800-晚報YT縮圖1-美伊開戰航母.txt' or slug '1800-晚報YT縮圖1-美伊開戰航母'


## 2. Simulate Original Filename Parsing & Logic

Simulate how the code currently handles the filename generation. The issue likely lies in how `generate_photoshop_script.py` uses the `slag` field. If `parse_file` fails to extract the slug correctly or if the variables passed to the generation function are empty, the filename construction `f"{mmdd}_{result_data['slag']}.psd"` might collapse to just `0130_.psd` (or similar).

In [3]:
def simulate_filename_generation(mmdd, slug, creator=""):
    def sanitize_filename(filename):
        invalid_chars = r'[<>:"/\\|?*]'
        filename = re.sub(invalid_chars, '_', filename)
        filename = filename.strip('. ')
        filename = re.sub(r'__+', '_', filename)
        return filename
    
    creator_suffix = f"_{creator.strip()}" if creator and creator.strip() else ""
    
    # This is the line from generate_photoshop_script.py (approx)
    # new_filename = sanitize_filename(f"{mmdd}_{result_data['slag']}{creator_suffix}.psd")
    
    raw_name_psd = f"{mmdd}_{slug}{creator_suffix}.psd"
    final_name_psd = sanitize_filename(raw_name_psd)
    
    # Simulate the JPG output name which usually follows the PSD name structure 
    # (Photoshop script saves as jpg using the document name)
    final_name_jpg = final_name_psd.replace(".psd", ".jpg")
    
    return final_name_jpg

# Test Case 1: Ideal Scenario
generated_ideal = simulate_filename_generation(mock_date, mock_slug)
print(f"Ideal generation result: {generated_ideal}")

# Test Case 2: Empty Slug (The suspected bug)
generated_bug = simulate_filename_generation(mock_date, "")
print(f"Empty slug generation result: {generated_bug}")
print(f"Matches incorrect filename? {generated_bug == incorrect_filename}") # 0130_.jpg vs 0130.jpg? Let's see sanitize logic.

Ideal generation result: 0130_1800-晚報YT縮圖1-美伊開戰航母.jpg
Empty slug generation result: 0130_.jpg
Matches incorrect filename? False


The analysis shows that if the slug is empty, we get `0130_.jpg`. However, the user reports `0130.jpg` (without the underscore).

This suggests that the `f"{mmdd}_{result_data['slag']}{creator_suffix}.psd"` logic might not be exactly what is happening, OR `mmdd` itself includes the underscore, OR the `sanitize_filename` function is stripping the trailing underscore.

Let's look at `sanitize_filename` again.
```python
        invalid_chars = r'[<>:"/\\|?*]'
        filename = re.sub(invalid_chars, '_', filename)
        filename = filename.strip('. ') # Strips dots and spaces
        filename = re.sub(r'__+', '_', filename)
```
If we have `0130_.psd`, `re.sub(r'__+', '_', filename)` doesn't remove the trailing underscore.

Let's test if `strip('. ')` is doing more than we think, or if my mock is slightly off.

Wait, if `result['slag']` is `None` or empty string, and the code is:
`f"{mmdd}_{result_data['slag']}{creator_suffix}.psd"` -> `0130_.psd`
Sanitize: `0130_.psd` -> `0130_.psd`
Strip: `0130_.psd` -> `0130_.psd` (strip only removes from ends)

Hypothesis: The variable `result_data['slag']` might not just be empty, code logic might be skipping the underscore if slag is missing?

Let's check `gui_main.py`'s `on_reprocess_requested`. It instantiates a `GenerationWorker` with `[filename]`.
The `GenerationWorker` calls `generate_jsx`.
`generate_jsx` calls `run_generation_logic`.
`run_generation_logic` calls `parse_file`.

If `parse_file` fails to read the Slag (first line) correctly, `result['slag']` is empty string.

But where does the underscore come from?
line 147 in `generate_photoshop_script.py`:
`new_filename = sanitize_filename(f"{mmdd}_{result_data['slag']}{creator_suffix}.psd")`

If slag is empty: `0130_+creator.psd`.

What if the underscore is removed by `sanitize_filename`?
If `0130` is the `mmdd`, and we have `0130_.psd`.
Maybe `re.sub(invalid_chars, '_', filename)` is fine.
Maybe `filename.strip('. ')` is fine.

Let's look at the `gui_main.py` reprocess logic again.
It calls `GenerationWorker` with `[filename]`.
Wait, the `filename` in `GenerationWorker` is just the filename (e.g. `1800-晚報YT縮圖1-美伊開戰航母.txt`).
`generate_jsx` joins `self.folder_path` and `filename`.
`self.folder_path` comes from `last_folder`.

If `parse_file` works, it reads the first line.

Maybe `parse_file` is NOT working for the file in question?
If `parse_file` fails (returns None), `run_generation_logic` returns 1.
Then the file generation should fail, not produce a generic filename.

**Alternative Theory**: The user is clicking "Reprocess". The logic for reprocess might be using a different path in `gui_main.py`.

In `gui_main.py`:
```python
            # 建立只包含這個檔案的 worker
            date = self.date_input.text().strip()
            creator = self.creator_input.text().strip()
            
            self.reprocess_worker = GenerationWorker([filename], date, creator, last_folder)
```
`date_input` text. Is it "0130"? Yes.

If `result['slag']` is empty, we get `0130_.jpg`.
The user says `0130.jpg`.

Maybe `invalid_chars` includes `_` in some version? No.

Let's try to pass `result['slag']` as `None`.
`f"{mmdd}_{None}.psd"` -> `0130_None.psd`.

Let's try to verify if `sanitize_filename` behavior changes if the string ends with `_`.
`filename.strip('. ')`. No `_` there.

Wait, look at `sanitize_filename` in `generate_photoshop_script.py`:
```python
        filename = re.sub(invalid_chars, '_', filename)
        filename = filename.strip('. ')
        filename = re.sub(r'__+', '_', filename)
```

If we have `0130_.psd`.
`__+` -> `_`. `0130_.psd`.

Is it possible `result_data['slag']` is causing an issue?

Let's look at `parse_file` in `parse_thumbnail_txt.py` again.
It reads the file using `utf-8`.
If the file encoded differently (e.g. BIG5)?
`parse_file` catches exception and prints error, returns `None`.
If `parse_file` returns `None`, `run_generation_logic` returns 1.
So we shouldn't get a file at all.

UNLESS: `lines` is empty list?
```python
        if lines:
            result["slag"] = lines[0].strip()
```
If `lines` is empty, `result['slag']` remains `""`.
Then we get `0130_.psd`.

So why `0130.jpg`?
Maybe `mmdd` is empty? No, it's 0130.

Maybe the `underscore` is NOT there if `slag` is empty?
The code is explicitly `f"{mmdd}_{result_data['slag']}..."`.

Let's verify the user's claim about `0130.jpg`.
Maybe they ignored the underscore or it's hard to see?
Or maybe `sanitize_filename` implementation changed?

Let's check the file `generate_photoshop_script.py` again.
I will read lines 130-150 again to be absolutely sure.

The code is definitely: `f"{mmdd}_{result_data['slag']}{creator_suffix}.psd"`.

If `slag` is empty, it produces `0130_.psd`.
If the user truly sees `0130.jpg`, then either:
1. `slag` is actually causing a backspace character? (Unlikely)
2. The user is imprecise (it's actually `0130_.jpg`).
3. My understanding of `sanitize_filename` regex is incomplete.

Let's test `re.sub(r'__+', '_', filename)` with `0130_.psd`. It should stay `0130_.psd`.

However, the problem is definitely that `slag` is empty or incorrectly retrieved.
If `slag` was retrieved correctly (`1800-晚報YT縮圖1-美伊開戰航母`), the filename would be correct.

**Root cause hypothesis**: `parse_file` is failing to get the slag.
Why?
Maybe `generate_photoshop_script.py` is being run such that it cannot read the file?
Or encoding issues.

The file `1800-晚報YT縮圖1-美伊開戰航母.txt` likely contains valid text.
The filename itself (minus .txt) is often the Slag, but the code relies on the first line of content.

Wait! In `gui_main.py`:
`existing_jpgs[norm_name] = f`

If the user sees `0130.jpg`, that means `sanitize_filename` produced `0130.psd`?
How?
`mmdd + "_" + slug`.
If `slug` starts with a character that `sanitize_filename` removes? No.

What if `mmdd` is empty?
`_slug.psd`.

What if `result['slag']` is being sanitized BEFORE this steps? No, it's right there.

Let's assume the user is reporting `0130.jpg` but it might be `0130_.jpg` or related to how they see it.
The core issue is that `result_data['slag']` is missing.

Why would `parse_file` return empty slag?
1. File is empty.
2. File encoding is not UTF-8 (e.g. `cp950` or `big5` common in TW Windows), and `utf-8` read fails or produces garbage that gets stripped?
3. `parse_file` logic has a bug.

Let's look at `parse_thumbnail_txt.py` again.
```python
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = [line.rstrip('\n\r') for line in f.readlines()]
```
If the file is Big5, `read()` with `utf-8` will throw `UnicodeDecodeError`.
The exception handler catches it:
```python
    except Exception as e:
        print(f"解析檔案時發生錯誤: {e}")
        return None
```
If it returns `None`, `run_generation_logic` returns 1.
`generate_jsx` in `worker.py`:
```python
            if result_code != 0:
                self.log.emit(f"⚠️ JSX 生成返回錯誤代碼: {result_code}")
            else:
                self.log.emit(f"✓ JSX 生成成功")
```
It logs failure. But it DOES NOT stop the flow inside `worker.py` effectively?
Wait, `run_generation_logic` is called.
If it fails, `latest_jsx` detection logic runs:
```python
            # 獲取生成的 JSX 路徑 ...
            # 找到最新的 JSX 文件
```
If generation failed, there is no new JSX. It might pick up an OLD JSX from the desktop folder?
OR if `run_generation_logic` successfully RAN but failed to parse, did it generate a script?
`run_generation_logic`:
```python
    result = parse_file(file_path)
    if not result:
        print("解析失敗")
        return 1
```
It returns 1. So no JSX is generated (the function returns before calling `generate_jsx_script`?).
Let's check `run_generation_logic` flow.

I need to read `generate_photoshop_script.py`'s `run_generation_logic` fully to see if it generates anything on error.

If `generate_photoshop_script.py` executes successfully (return 0), it implies `result['slag']` was printed and was ostensibly valid.

However, in `gui_main.py`:
```python
            # 建立只包含這個檔案的 worker
            self.reprocess_worker = GenerationWorker([filename], date, creator, last_folder)
```
The `GenerationWorker` receives `checked_files` as a list.
In `worker.py`, `generate_jsx` runs:
```python
            result_code = run_generation_logic(
                str(file_path),
                None, # color_id
                ...
            )
```
`file_path` is constructed in `worker.py`: `file_path = os.path.join(self.folder_path, filename)`.

Wait! The user says: "新生成的檔名不是0130_1800-晚報YT縮圖1-美伊開戰航母.jpg 而是0130.jpg"

If the user sees `0130.jpg`, it might be that `result_data['slag']` was EMPTY, and somehow `sanitize_filename` caused `0130_.jpg` to become `0130.jpg`?
NO, I tested that.

Is it possible `mmdd` is missing? No, date is 0130.

Is it possible that the file they are looking at is NOT generated by the script, but by something else?
The user clicked "Reprocess" (刷新/重整).

Maybe the `GenerationWorker` logic or `gui_main.py` logic has a flaw when `reprocess` is called.

In `gui_main.py`, `on_reprocess_requested`:
```python
            # 建立只包含這個檔案的 worker
            date = self.date_input.text().strip()  # <--- Date from Input
            creator = self.creator_input.text().strip()
            last_folder = self.settings.get("last_folder", os.getcwd())
            
            self.reprocess_worker = GenerationWorker([filename], date, creator, last_folder)
```
In `worker.py`:
```python
    def __init__(self, checked_files, date, creator, folder_path, parent=None):
        # ...
        self.date = date if date else get_today_mmdd()
```
If `date` input is empty, it uses today's date.

Let's check `parse_thumbnail_txt.py` result again.
If `parse_file` returns empty slag, `result['slag']` is `""`.

Why is `result['slag']` empty?
The most likely reason is encoding.
If `1800-晚報YT縮圖1-美伊開戰航母.txt` is encoded in ANSI (CP950 on TW Windows), and we read with `utf-8`.
`utf-8` reading CP950 might result in:
1. Decoding error (caught -> None -> failure).
2. Successfully decoded but garbage characters.

If garbage characters, `result['slag']` would be garbage.
`new_filename` would be `0130_GARBAGE.psd`.
`sanitize_filename` replaces garbage with `_`.
`0130__.psd` -> `0130_.psd`.

Still, where does `0130.jpg` come from?

Maybe the `slag` line starts with dots or spaces?
`lines[0].strip()`.

If the file content is empty?
`if lines:` check fails. `result['slag']` is `""`.

I suspect the user might be misreporting `0130.jpg` vs `0130_.jpg`.
But I should focus on WHY `slag` is empty.

If I can fix `parse_file` to support CP950/Big5, I fix the root cause.

Let's modify `parse_thumbnail_txt.py` to handle encoding detection.

But first, let's verify if that's the issue.
I can check `parse_thumbnail_txt.py` code again.
It strictly uses `encoding='utf-8'`.
On Windows with `chcp 65001` (UTF-8), Python usually defaults to UTF-8 or system locale?
The script explicitly sets `sys.stdout` to utf-8.
But `open(file_path, 'r', encoding='utf-8')` is explicit.

If the user created the txt file using Notepad on Windows (Traditional Chinese), it defaults to ANSI (CP950) or UTF-8 with BOM.
If it is CP950, `utf-8` decoding will fail for Chinese characters.

If `parse_file` returns `None`, `run_generation_logic` returns 1.
`GenerationWorker` logs error.
NO JSX is generated.
NO Photoshop execution occurs.
So `0130.jpg` should NOT be generated.

Unless... `run_photoshop_jsx` inside `worker.py` runs even if generation failed?
```python
            # 第 1 階段：生成所有 JSX 腳本
            for index, filename in enumerate(self.checked_files):
                # ...
                jsx_path = self.generate_jsx_with_path(file_path)
                if jsx_path:
                   jsx_map[filename] = jsx_path
                else:
                   # ... failed_count += 1
```
If `jsx_map` is empty:
```python
            if not jsx_map:
                self.log.emit("❌ 未能生成任何 JSX 文件")
                self.completed.emit(0, total_count, total_count)
                return
```
So execution stops.

So `run_generation_logic` MUST be returning success (0).
This means `parse_file` SUCCEEDED.
This means `result['slag']` was extracted.

If `result['slag']` was extracted, why is filename truncated?
Maybe the first line of the text file is EMPTY?
`lines[0].strip()`.
If the file starts with an empty line?
`lines[0]` is the first line. If it's `\n`, stripped is `""`.

If the text file looks like:
```text

主播: ...
...
```
Then `slag` is `""`.
And `generate_photoshop_script.py` produces `0130_.psd`.

So, the fix is to search for a non-empty line as slag, or use the filename as fallback if slag is empty.
The user's file is named `1800-晚報YT縮圖1-美伊開戰航母.txt`.
Using the filename as fallback for Slag seems like the robust solution requested.

The user asked: "新生成的檔名不是... 而是0130.jpg".
This strongly implies the suffix (Slag) is missing.

I will modifying `parse_thumbnail_txt.py` to use filename as fallback if the first line is empty or missing.
AND I will modify `generate_photoshop_script.py` to fallback to filename if `result['slag']` is empty.

Wait, `generate_photoshop_script.py` sanitization logic:
```python
    def sanitize_filename(filename):
        # ...
        filename = re.sub(invalid_chars, '_', filename)
        filename = filename.strip('. ') # <--- this removes trailing dots and spaces
        filename = re.sub(r'__+', '_', filename)
```
If the result is `0130_.psd`. `strip('. ')` does NOT remove `_`.

So if the user really sees `0130.jpg`, maybe `sanitize_filename` is buggy in the user's version?
I checked the file content I read earlier. It looks correct.

Regardless, I'll implement the fallback logic. This should solve the "missing slag" issue.

**Plan:**
1.  Modify `parse_thumbnail_txt.py`: `parse_file` should accept the filename as a hint? Or just return what it finds.
2.  Modify `generate_photoshop_script.py`: `run_generation_logic` should check if `result['slag']` is empty. If so, use the base filename (minus extension) as the Slag.
3.  Also Update `parse_thumbnail_txt.py` to try different encodings (utf-8, cp950) just in case.

This two-pronged approach ensures we get text, and even if we don't get text from the file content, we use the filename.
This perfectly addresses "why... new filename is 0130.jpg" -> "metadata missing".

Let's verify `utf-8` vs `cp950` first?
I'll add the encoding fallback to `parse_thumbnail_txt.py`.
I'll add the filename fallback to `parse_thumbnail_txt.py` (it's the parser's job to provide the slag).
Actually, `parse_file` takes `file_path`. It can extract the filename itself.

In `parse_thumbnail_txt.py`:
```python
def parse_file(file_path):
    # ...
    # 1. 第一行定義為Slag
    if lines and lines[0].strip():
        result["slag"] = lines[0].strip()
    else:
        # Fallback to filename
        result["slag"] = os.path.splitext(os.path.basename(file_path))[0]
```
This is a very safe fix.

One more thing. `generate_photoshop_script.py` sanitization.
If I put the filename back in, it will be sanitized anyway.
`1800-晚報...` -> `1800-晚報...` (valid chars).

Let's do this.